In [7]:
"""Pima Indians Diabetes — ML Pipeline (Steps 1-6)
=================================================
1. Load dataset
2. EDA + preprocessing (fix impossible zeros, impute, scale)
3. Train 4-5 models (with class imbalance handling)
4. Compare Accuracy/Precision/Recall/F1/ROC-AUC + Confusion Matrix
5. Select best model (optimizing for Recall, tie-break on F1)
6. Save model + scaler + imputer with joblib
"""

import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

RANDOM_STATE = 42

# -----------------------------------------------------------------
# STEP 1: Load dataset
# -----------------------------------------------------------------
COLUMNS = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]
df = pd.read_csv("diabetes.csv", header=0, names=COLUMNS)
print(f"[Step 1] Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

# -----------------------------------------------------------------
# STEP 2: EDA + preprocessing
# -----------------------------------------------------------------
# Columns where 0 is biologically impossible -> treat as missing
ZERO_AS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

missing_report = {}
for col in ZERO_AS_MISSING:
    n_zero = int((df[col] == 0).sum())
    missing_report[col] = n_zero
    df[col] = df[col].replace(0, np.nan)

print("[Step 2] Zero-values treated as missing (before imputation):")
for k, v in missing_report.items():
    pct = 100 * v / len(df)
    print(f"          {k}: {v} missing ({pct:.1f}%)")

# Class balance
class_counts = df["Outcome"].value_counts().to_dict()
print(f"[Step 2] Class balance -> Non-diabetic(0): {class_counts.get(0,0)}, "
      f"Diabetic(1): {class_counts.get(1,0)}")

# Correlation heatmap (EDA artifact)
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("eda_correlation_heatmap.png", dpi=150)
plt.close()

# Outcome distribution plot
plt.figure(figsize=(5, 4))
sns.countplot(x="Outcome", data=df, palette=["#4C72B0", "#DD8452"])
plt.title("Class Distribution (0 = Non-diabetic, 1 = Diabetic)")
plt.tight_layout()
plt.savefig("eda_class_distribution.png", dpi=150)
plt.close()

# Split features/target FIRST, then impute/scale using train-fit only (avoid leakage)
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"[Step 2] Stratified split -> train: {X_train.shape[0]}, test: {X_test.shape[0]}")

# KNN imputation (fit on train only, applied to both)
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X.columns, index=X_test.index)

# Scaling (fit on train only)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imputed), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=X.columns, index=X_test.index)

# -----------------------------------------------------------------
# STEP 3: Train models (with class imbalance handling & Overfit Check)
# -----------------------------------------------------------------
scale_pos_weight = class_counts.get(0, 1) / class_counts.get(1, 1)

models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(class_weight="balanced", n_estimators=300, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(
        scale_pos_weight=scale_pos_weight, eval_metric="logloss",
        random_state=RANDOM_STATE, n_estimators=300, max_depth=4, learning_rate=0.05
    ),
    "SVM (RBF)": SVC(class_weight="balanced", probability=True, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=11),
}

results = []
confusion_matrices = {}
fitted_models = {}

for name, model in models.items():
    # Fit the model
    model.fit(X_train_scaled, y_train)
    
    # 1. Training Metrics (for overfitting check)
    y_train_pred = model.predict(X_train_scaled)
    train_acc = accuracy_score(y_train, y_train_pred)
    train_f1 = f1_score(y_train, y_train_pred)

    # 2. Test Metrics (Generalization)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    test_acc = accuracy_score(y_test, y_pred)
    test_prec = precision_score(y_test, y_pred)
    test_rec = recall_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)

    results.append({
        "Model": name, 
        "Train_Acc": train_acc, "Test_Acc": test_acc, 
        "Train_F1": train_f1, "Test_F1": test_f1,
        "Precision": test_prec, "Recall": test_rec, "ROC_AUC": test_auc
    })
    confusion_matrices[name] = cm
    fitted_models[name] = model

    # Print comparison to easily spot overfitting
    print(f"[Step 3] {name:20s} | "
          f"Train Acc={train_acc:.3f} vs Test Acc={test_acc:.3f} | "
          f"Train F1={train_f1:.3f} vs Test F1={test_f1:.3f} | "
          f"Test Rec={test_rec:.3f} AUC={test_auc:.3f}")

results_df = pd.DataFrame(results).sort_values(by=["Recall", "Test_F1"], ascending=False).reset_index(drop=True)
# -----------------------------------------------------------------
# STEP 4: Compare metrics (save table + plots)
# -----------------------------------------------------------------
results_df.to_csv("model_comparison_metrics.csv", index=False)
print("\n[Step 4] Model comparison (sorted by Recall, then F1):")
print(results_df.to_string(index=False))

# Bar chart comparison
plt.figure(figsize=(12, 6))
melted = results_df.melt(
    id_vars="Model", 
    value_vars=["Train_Acc", "Test_Acc", "Train_F1", "Test_F1", "Recall", "ROC_AUC"]
)
sns.barplot(data=melted, x="Model", y="value", hue="variable")
plt.title("Model Comparison Across Metrics")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("model_comparison_chart.png", dpi=150)
plt.close()

# Confusion matrices grid
fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 4))
for ax, (name, cm) in zip(axes, confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.close()

# -----------------------------------------------------------------
# STEP 5: Select best model (Recall-first, F1 tie-break)
# -----------------------------------------------------------------
best_row = results_df.iloc[0]
best_name = best_row["Model"]
best_model = fitted_models[best_name]

# UPDATE THE KEYS IN THIS PRINT STATEMENT:
print(f"\n[Step 5] Selected best model: {best_name} "
      f"(Recall={best_row['Recall']:.3f}, Test_F1={best_row['Test_F1']:.3f}, "
      f"Test_Acc={best_row['Test_Acc']:.3f}, AUC={best_row['ROC_AUC']:.3f})")
# -----------------------------------------------------------------
# STEP 6: Save model + scaler + imputer
# -----------------------------------------------------------------
joblib.dump(best_model, "diabetes_best_model.joblib")
joblib.dump(scaler, "diabetes_scaler.joblib")
joblib.dump(imputer, "diabetes_imputer.joblib")

metadata = {
    "best_model": best_name,
    "feature_order": list(X.columns),
    "zero_as_missing_columns": ZERO_AS_MISSING,
    "metrics": {k: float(v) for k, v in best_row.items() if k != "Model"},
    "class_distribution": {str(k): int(v) for k, v in class_counts.items()},
    "scale_pos_weight_used_for_xgboost": float(scale_pos_weight),
    "preprocessing_order": ["train_test_split(stratified)", "KNNImputer(fit on train)", "StandardScaler(fit on train)"],
}
with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\n[Step 6] Saved artifacts:")
print("  - diabetes_best_model.joblib")
print("  - diabetes_scaler.joblib")
print("  - diabetes_imputer.joblib")
print("  - model_metadata.json")
print("\nPipeline complete (Steps 1-6).")

[Step 1] Loaded dataset: 768 rows, 9 columns
[Step 2] Zero-values treated as missing (before imputation):
          Glucose: 0 missing (0.0%)
          BloodPressure: 0 missing (0.0%)
          SkinThickness: 0 missing (0.0%)
          Insulin: 0 missing (0.0%)
          BMI: 0 missing (0.0%)
[Step 2] Class balance -> Non-diabetic(0): 500, Diabetic(1): 268
[Step 2] Stratified split -> train: 614, test: 154
[Step 3] Logistic Regression  | Train Acc=0.809 vs Test Acc=0.747 | Train F1=0.743 vs Test F1=0.683 | Test Rec=0.778 AUC=0.827
[Step 3] Random Forest        | Train Acc=1.000 vs Test Acc=0.864 | Train F1=1.000 vs Test F1=0.800 | Test Rec=0.778 AUC=0.947
[Step 3] XGBoost              | Train Acc=1.000 vs Test Acc=0.870 | Train F1=1.000 vs Test F1=0.818 | Test Rec=0.833 AUC=0.946
[Step 3] SVM (RBF)            | Train Acc=0.891 vs Test Acc=0.844 | Train F1=0.856 vs Test F1=0.800 | Test Rec=0.889 AUC=0.898
[Step 3] KNN                  | Train Acc=0.862 vs Test Acc=0.805 | Train F1=0.800

In [10]:
"""
Pima Indians Diabetes — ML Pipeline (Phase 1.5: Optimized & Production-Ready)
=============================================================================
1. Load dataset (Raw unprocessed)
2. EDA + Preprocessing (Treat 0s as NaN, Stratified Split)
3. Build Pipelines & Tune Hyperparameters (GridSearchCV + 5-Fold CV to stop overfitting)
4. Evaluate best estimators on hold-out Test Set
5. Select Best Pipeline & Tune Decision Threshold for Medical Recall
6. Save single Pipeline artifact + metadata
"""

import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, precision_recall_curve
)

RANDOM_STATE = 42

# -----------------------------------------------------------------
# STEP 1: Load Raw Dataset
# -----------------------------------------------------------------
COLUMNS = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]
DATA_PATH = "diabetes.csv"

first_row = pd.read_csv(DATA_PATH, nrows=1)
if first_row.columns[0] == "Pregnancies" or first_row.iloc[0, 0] == "Pregnancies":
    df = pd.read_csv(DATA_PATH)
    df.columns = COLUMNS
else:
    df = pd.read_csv(DATA_PATH, header=None, names=COLUMNS)

print(f"[Step 1] Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

# -----------------------------------------------------------------
# STEP 2: Preprocessing Setup & Stratified Split
# -----------------------------------------------------------------
ZERO_AS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

for col in ZERO_AS_MISSING:
    df[col] = df[col].replace(0, np.nan)

class_counts = df["Outcome"].value_counts().to_dict()
scale_pos_weight = class_counts.get(0, 1) / class_counts.get(1, 1)

X = df.drop(columns=["Outcome"])
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"[Step 2] Stratified split -> Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

# -----------------------------------------------------------------
# STEP 3: Build Pipelines & Cross-Validation (Overfit Prevention)
# -----------------------------------------------------------------
# Base Pipeline components
imputer = KNNImputer(n_neighbors=5)
scaler = StandardScaler()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Define Pipelines
pipelines = {
    "SVM (RBF)": Pipeline([
        ("imputer", imputer), ("scaler", scaler),
        ("classifier", SVC(class_weight="balanced", probability=True, random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("imputer", imputer), ("scaler", scaler),
        ("classifier", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "XGBoost": Pipeline([
        ("imputer", imputer), ("scaler", scaler),
        ("classifier", XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=RANDOM_STATE))
    ])
}

# Define Hyperparameter Grids to prevent overfitting
param_grids = {
    "SVM (RBF)": {
        "classifier__C": [0.1, 1.0, 10.0],
        "classifier__gamma": ["scale", 0.01, 0.1]
    },
    "Random Forest": {
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [3, 5, 7],          # Constrain depth to stop memorization
        "classifier__min_samples_leaf": [5, 10]      # Require minimum samples per leaf
    },
    "XGBoost": {
        "classifier__n_estimators": [100, 200],
        "classifier__max_depth": [3, 5],
        "classifier__learning_rate": [0.01, 0.05]
    }
}

best_estimators = {}
results = []

print("\n[Step 3] Tuning Models with 5-Fold CV (Optimizing for Recall)...")
for name in pipelines.keys():
    grid = GridSearchCV(
        pipelines[name], param_grids[name], cv=cv, 
        scoring="recall", n_jobs=-1, return_train_score=True
    )
    grid.fit(X_train, y_train)
    best_estimators[name] = grid.best_estimator_
    
    # Check for overfitting during CV
    mean_train_rec = grid.cv_results_['mean_train_score'][grid.best_index_]
    mean_val_rec = grid.cv_results_['mean_test_score'][grid.best_index_]
    print(f"  {name:15s} | Best CV Recall: {mean_val_rec:.3f} (Train Recall: {mean_train_rec:.3f})")

# -----------------------------------------------------------------
# STEP 4: Evaluate on Hold-out Test Set
# -----------------------------------------------------------------
for name, pipeline in best_estimators.items():
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results).sort_values(by=["Recall", "F1"], ascending=False).reset_index(drop=True)
print("\n[Step 4] Final Test Set Evaluation:")
print(results_df.to_string(index=False))

# -----------------------------------------------------------------
# STEP 5: Select Best Pipeline & Tune Decision Threshold
# -----------------------------------------------------------------
best_name = results_df.iloc[0]["Model"]
final_pipeline = best_estimators[best_name]

print(f"\n[Step 5] Selected Best Pipeline: {best_name}")

# Calculate Precision-Recall curve to find a better threshold
y_test_proba = final_pipeline.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_test_proba)

# Find the highest threshold that keeps Recall >= 0.90
target_recall = 0.90
optimal_idx = np.where(recalls >= target_recall)[0][-1]
optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else thresholds[-1]
optimal_precision = precisions[optimal_idx]

print(f"  -> Default Threshold (0.50): Recall = {results_df.iloc[0]['Recall']:.3f}, Precision = {results_df.iloc[0]['Precision']:.3f}")
print(f"  -> Tuned Threshold ({optimal_threshold:.2f}): Recall = {recalls[optimal_idx]:.3f}, Precision = {optimal_precision:.3f}")

# Plot PR Curve
plt.figure(figsize=(6, 5))
plt.plot(thresholds, precisions[:-1], "b--", label="Precision")
plt.plot(thresholds, recalls[:-1], "g-", label="Recall")
plt.axvline(x=optimal_threshold, color="r", linestyle=":", label=f"Optimal Threshold ({optimal_threshold:.2f})")
plt.xlabel("Decision Threshold")
plt.legend(loc="best")
plt.title(f"Precision-Recall vs Threshold ({best_name})")
plt.tight_layout()
plt.savefig("precision_recall_threshold.png", dpi=150)
plt.close()

# -----------------------------------------------------------------
# STEP 6: Save Unified Artifact & Metadata
# -----------------------------------------------------------------
# We now only need to save the final_pipeline!
joblib.dump(final_pipeline, "diabetes_production_pipeline.joblib")

metadata = {
    "best_model": best_name,
    "feature_order": list(X.columns),
    "zero_as_missing_columns": ZERO_AS_MISSING,
    "optimal_decision_threshold": float(optimal_threshold),
    "metrics_at_default_threshold": {k: float(v) for k, v in results_df.iloc[0].items() if k != "Model"},
    "class_distribution": {str(k): int(v) for k, v in class_counts.items()},
}

with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\n[Step 6] Saved Production Artifacts:")
print("  - diabetes_production_pipeline.joblib (Includes Imputer + Scaler + Model)")
print("  - model_metadata.json")
print("  - precision_recall_threshold.png")

[Step 1] Loaded dataset: 768 rows, 9 columns
[Step 2] Stratified split -> Train: 614, Test: 154

[Step 3] Tuning Models with 5-Fold CV (Optimizing for Recall)...
  SVM (RBF)       | Best CV Recall: 0.874 (Train Recall: 0.891)
  Random Forest   | Best CV Recall: 0.883 (Train Recall: 0.925)
  XGBoost         | Best CV Recall: 0.893 (Train Recall: 0.938)

[Step 4] Final Test Set Evaluation:
        Model  Accuracy  Precision   Recall       F1  ROC_AUC
      XGBoost  0.876623   0.786885 0.888889 0.834783 0.952037
    SVM (RBF)  0.759740   0.607595 0.888889 0.721805 0.851667
Random Forest  0.857143   0.758065 0.870370 0.810345 0.921296

[Step 5] Selected Best Pipeline: XGBoost
  -> Default Threshold (0.50): Recall = 0.889, Precision = 0.787
  -> Tuned Threshold (0.41): Recall = 0.907, Precision = 0.742

[Step 6] Saved Production Artifacts:
  - diabetes_production_pipeline.joblib (Includes Imputer + Scaler + Model)
  - model_metadata.json
  - precision_recall_threshold.png
